# Notebook 4 — Mixed-Effects Regression: Surprisal as a Predictor of Reading Time

This notebook fits **linear mixed-effects regression** models of the form:

```
log(RT) ~ surprisal + word_len + log_freq + sentence_position + (1|subject) + (1|item)
```

Goals:
- Estimate the **β-coefficient** of surprisal (bits → ms effect on log RT)
- Compare β across all four models (Bigram, Trigram, GPT-2, BERT)
- Confirm surprisal effect survives control variables (partial regression / LMER-style)
- Answer **RQ1** more rigorously: effect size, not just correlation

> **Inputs**: outputs of Notebooks 1–3  
> **Outputs**: `results/nb4_lmer_coefficients.csv`, `results/report_figures/fig_nb4_*.png`


In [1]:
import os, re, sys, subprocess, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
matplotlib.rcParams.update({"font.size": 12, "figure.dpi": 120})

# ── Install statsmodels (for OLS + partial regression) ────────────────────────
try:
    import statsmodels.formula.api as smf
    import statsmodels.api as sm
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "statsmodels", "--quiet"])
    import statsmodels.formula.api as smf
    import statsmodels.api as sm

# ── Path resolver (identical pattern to notebooks 2 & 3) ─────────────────────
def resolve_data_paths():
    kaggle_input = "/kaggle/input"
    if os.path.exists(kaggle_input):
        for root, dirs, _ in os.walk(kaggle_input):
            if "natural_stories" in set(dirs) and "dundee" in set(dirs):
                return (os.path.join(root, "natural_stories"),
                        os.path.join(root, "dundee"),
                        "/kaggle/working/results", "kaggle")
            if "data" in set(dirs):
                dr = os.path.join(root, "data")
                ns, du = os.path.join(dr, "natural_stories"), os.path.join(dr, "dundee")
                if os.path.exists(ns) and os.path.exists(du):
                    return ns, du, "/kaggle/working/results", "kaggle"
    base = os.path.abspath(os.path.join(os.getcwd(), ".."))
    return (os.path.join(base, "data", "natural_stories"),
            os.path.join(base, "data", "dundee"),
            os.path.join(base, "results"), "local")

DATA_NS, DATA_DU, RESULTS, ENV = resolve_data_paths()
FIG_DIR = os.path.join(RESULTS, "report_figures")
OUT_DIR = os.path.join(RESULTS, "notebook_outputs")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

print(f"Environment : {ENV}")
print(f"Results dir : {RESULTS}")


Environment : local
Results dir : c:\Users\mudit\OneDrive\Desktop\Fourth Sem\computational-psycholinguistics\Project\results


---
## Part 1: Load & Merge All Surprisal Sources


In [3]:
# ── Load word-level aggregates (Notebook 1) ───────────────────────────────────
ns_agg  = pd.read_csv(os.path.join(DATA_NS, "ns_word_agg.csv"))
du_agg  = pd.read_csv(os.path.join(DATA_DU, "dundee_word_agg.csv"))

# ── Resolve path to result files (may be in subdirectories from prior runs) ────
def find_result_file(filename):
    """Search for result file in RESULTS dir and known subdirectories."""
    candidates = [
        os.path.join(RESULTS, filename),
        os.path.join(RESULTS, "notebook 2 results", "results", filename),
        os.path.join(RESULTS, "Notebook 3 results", "results", filename),
    ]
    for path in candidates:
        if os.path.exists(path):
            print(f"Found {filename} at: {path}")
            return path
    raise FileNotFoundError(f"Could not find {filename} in:\n  " + "\n  ".join(candidates))

# ── Load n-gram surprisal (Notebook 2) ────────────────────────────────────────
ns_ngram = pd.read_csv(find_result_file("ns_ngram_surprisal.csv"))
du_ngram = pd.read_csv(find_result_file("dundee_ngram_surprisal.csv"))

# ── Load transformer surprisal (Notebook 3) ───────────────────────────────────
ns_gpt2  = pd.read_csv(find_result_file("ns_gpt2_surprisal.csv"))
ns_bert  = pd.read_csv(find_result_file("ns_bert_surprisal.csv"))
du_gpt2  = pd.read_csv(find_result_file("dundee_gpt2_surprisal.csv"))
du_bert  = pd.read_csv(find_result_file("dundee_bert_surprisal.csv"))

# ── Merge into one flat frame per corpus ──────────────────────────────────────
ns_full = (
    ns_agg
    .merge(ns_ngram[["story","zone","bigram_surprisal","trigram_surprisal"]],
           on=["story","zone"], how="inner")
    .merge(ns_gpt2[["story","zone","gpt2_surprisal"]],  on=["story","zone"], how="inner")
    .merge(ns_bert[["story","zone","bert_surprisal"]],   on=["story","zone"], how="inner")
)

du_full = (
    du_agg
    .merge(du_ngram[["text_num","WNUM","bigram_surprisal","trigram_surprisal"]],
           on=["text_num","WNUM"], how="inner")
    .merge(du_gpt2[["text_num","WNUM","gpt2_surprisal"]], on=["text_num","WNUM"], how="inner")
    .merge(du_bert[["text_num","WNUM","bert_surprisal"]],  on=["text_num","WNUM"], how="inner")
)

print(f"NS  full frame : {len(ns_full):,} rows × {ns_full.shape[1]} cols")
print(f"DU  full frame : {len(du_full):,} rows × {du_full.shape[1]} cols")
ns_full.head(5)

Found ns_ngram_surprisal.csv at: c:\Users\mudit\OneDrive\Desktop\Fourth Sem\computational-psycholinguistics\Project\results\Notebook 3 results\results\ns_ngram_surprisal.csv
Found dundee_ngram_surprisal.csv at: c:\Users\mudit\OneDrive\Desktop\Fourth Sem\computational-psycholinguistics\Project\results\Notebook 3 results\results\dundee_ngram_surprisal.csv
Found ns_gpt2_surprisal.csv at: c:\Users\mudit\OneDrive\Desktop\Fourth Sem\computational-psycholinguistics\Project\results\notebook 2 results\results\ns_gpt2_surprisal.csv
Found ns_bert_surprisal.csv at: c:\Users\mudit\OneDrive\Desktop\Fourth Sem\computational-psycholinguistics\Project\results\notebook 2 results\results\ns_bert_surprisal.csv
Found dundee_gpt2_surprisal.csv at: c:\Users\mudit\OneDrive\Desktop\Fourth Sem\computational-psycholinguistics\Project\results\notebook 2 results\results\dundee_gpt2_surprisal.csv
Found dundee_bert_surprisal.csv at: c:\Users\mudit\OneDrive\Desktop\Fourth Sem\computational-psycholinguistics\Project\r

,story,zone,word_text_rt,mean_RT,mean_log_RT,mean_log_RT_z,n_subjects,word_text,word_clean,word_len,bigram_surprisal,trigram_surprisal,gpt2_surprisal,bert_surprisal
0,1,1,If,578.964286,6.268741,2.052692,84,If,If,2,10.104788,10.104794,NaN,0.035766
1,1,2,you,369.011905,5.830609,0.378104,84,you,you,3,3.275795,4.337341,4.683665,0.006446
2,1,3,were,368.183908,5.828199,0.355197,87,were,were,4,5.726070,5.235687,5.995212,0.181674
3,1,4,to,344.318182,5.737662,0.017462,88,to,to,2,5.422809,5.962780,2.357792,0.002890
4,1,5,journey,354.639535,5.722561,-0.008469,86,journey,journey,7,14.795042,18.714258,14.247714,6.919935


---
## Part 2: Feature Engineering
Add log-frequency proxy and normalise zone position.


In [ ]:
# ── 2.1 Log word frequency proxy ──────────────────────────────────────────────
# True corpus frequency requires a frequency lexicon (e.g. SUBTLEX-US).
# Here we use Brown-corpus unigram log-frequency as a lightweight proxy —
# the same Brown corpus already used to train n-gram models in Notebook 2.
import nltk
try:
    from nltk.corpus import brown
except:
    nltk.download("brown", quiet=True)
    from nltk.corpus import brown

brown_tokens = [w.lower() for w in brown.words()]
from collections import Counter
brown_counts = Counter(brown_tokens)
total_brown  = sum(brown_counts.values())

def log_freq_proxy(word):
    if pd.isna(word) or not str(word).strip():
        return np.nan
    w = re.sub(r"[^a-z0-9'-]", "", str(word).lower())
    cnt = brown_counts.get(w, 1)          # Laplace floor
    return np.log(cnt / total_brown)       # negative; higher = more frequent

# Apply to both corpora
word_col_ns = "word_text" if "word_text" in ns_full.columns else "word"
word_col_du = "word_text" if "word_text" in du_full.columns else "word"

ns_full["log_freq"] = ns_full[word_col_ns].map(log_freq_proxy)
du_full["log_freq"] = du_full[word_col_du].map(log_freq_proxy)

# ── 2.2 Normalise zone / WNUM to sentence position (0-1 range per text) ───────
ns_full["position_norm"] = ns_full.groupby("story")["zone"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-8))

du_full["position_norm"] = du_full.groupby("text_num")["WNUM"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-8))

# ── 2.3 Item ID for random effects (story-zone / text-word) ───────────────────
ns_full["item_id"] = ns_full["story"].astype(str) + "_" + ns_full["zone"].astype(str)
du_full["item_id"] = du_full["text_num"].astype(str) + "_" + du_full["WNUM"].astype(str)

print("Feature columns added: log_freq, position_norm, item_id")
print()
print("NS  — log_freq summary:")
print(ns_full["log_freq"].describe().round(3).to_string())


---
## Part 3: OLS Regression per Model
We use **ordinary least squares** (OLS) as the workhorse — it gives identical β-estimates to mixed-effects when random effects are modest, and is fully available via `statsmodels`.

Model formula:
```
mean_log_RT ~ surprisal + word_len + log_freq + position_norm
```


In [ ]:
# ── 3.1 OLS regression helper ─────────────────────────────────────────────────
def run_ols(df, surp_col, dv_col, controls):
    """Fit OLS model; return result object and tidy coefficient table."""
    cols_needed = [surp_col, dv_col] + controls
    sub = df[cols_needed].dropna()
    formula_parts = [surp_col] + controls
    formula = f"{dv_col} ~ " + " + ".join(formula_parts)
    model = smf.ols(formula, data=sub).fit()
    coef = model.params
    ci   = model.conf_int()
    tidy = pd.DataFrame({
        "coef":  coef,
        "se":    model.bse,
        "t":     model.tvalues,
        "p":     model.pvalues,
        "ci_lo": ci[0],
        "ci_hi": ci[1],
    }).reset_index().rename(columns={"index": "term"})
    return model, tidy

CONTROLS_NS = ["word_len", "log_freq", "position_norm"]
CONTROLS_DU = ["word_len", "log_freq", "position_norm"]

MODEL_COLS = [
    ("Bigram KN",  "bigram_surprisal"),
    ("Trigram KN", "trigram_surprisal"),
    ("GPT-2",      "gpt2_surprisal"),
    ("BERT (PLL)", "bert_surprisal"),
]

# ── 3.2 Run for Natural Stories ───────────────────────────────────────────────
print("=" * 65)
print("NATURAL STORIES — OLS: log(RT) ~ surprisal + controls")
print("=" * 65)

ns_ols_rows = []
for model_name, col in MODEL_COLS:
    res, tidy = run_ols(ns_full, col, "mean_log_RT", CONTROLS_NS)
    row_surp  = tidy[tidy["term"] == col].iloc[0]
    ns_ols_rows.append({
        "Model":     model_name,
        "β_surprisal": row_surp["coef"],
        "SE":          row_surp["se"],
        "t":           row_surp["t"],
        "p":           row_surp["p"],
        "CI_lo":       row_surp["ci_lo"],
        "CI_hi":       row_surp["ci_hi"],
        "R²":          res.rsquared,
        "R²_adj":      res.rsquared_adj,
    })

ns_ols_table = pd.DataFrame(ns_ols_rows)
print(ns_ols_table.round(5).to_string(index=False))


In [ ]:
# ── 3.3 Run for Dundee ────────────────────────────────────────────────────────
print("=" * 65)
print("DUNDEE — OLS: log(FDUR) ~ surprisal + controls")
print("=" * 65)

du_ols_rows = []
for model_name, col in MODEL_COLS:
    res, tidy = run_ols(du_full, col, "mean_log_FDUR", CONTROLS_DU)
    row_surp  = tidy[tidy["term"] == col].iloc[0]
    du_ols_rows.append({
        "Model":       model_name,
        "β_surprisal": row_surp["coef"],
        "SE":          row_surp["se"],
        "t":           row_surp["t"],
        "p":           row_surp["p"],
        "CI_lo":       row_surp["ci_lo"],
        "CI_hi":       row_surp["ci_hi"],
        "R²":          res.rsquared,
        "R²_adj":      res.rsquared_adj,
    })

du_ols_table = pd.DataFrame(du_ols_rows)
print(du_ols_table.round(5).to_string(index=False))

# Save combined
ns_ols_table["Corpus"] = "Natural Stories"
du_ols_table["Corpus"] = "Dundee"
all_ols = pd.concat([ns_ols_table, du_ols_table], ignore_index=True)
all_ols.to_csv(os.path.join(RESULTS, "nb4_lmer_coefficients.csv"), index=False)
print("\nSaved: nb4_lmer_coefficients.csv")


---
## Part 4: Visualise β-Coefficients with 95% Confidence Intervals


In [ ]:
# ── 4.1 Forest plot of β_surprisal per model × corpus ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

colors = {"Natural Stories": "steelblue", "Dundee": "coral"}
tables = {"Natural Stories": ns_ols_table, "Dundee": du_ols_table}

for ax, (corpus, tbl) in zip(axes, tables.items()):
    y_pos  = list(range(len(tbl)))
    betas  = tbl["β_surprisal"].tolist()
    lo     = tbl["CI_lo"].tolist()
    hi     = tbl["CI_hi"].tolist()
    labels = tbl["Model"].tolist()
    errs   = [[b - l for b, l in zip(betas, lo)],
               [h - b for h, b in zip(hi, betas)]]
    ax.barh(y_pos, betas, xerr=errs, color=colors[corpus],
            align="center", alpha=0.8, capsize=5, ecolor="black")
    ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels)
    ax.set_xlabel("β coefficient (log RT / bit)")
    ax.set_title(f"{corpus}\nβ_surprisal  (95 % CI)")
    for i, (b, p) in enumerate(zip(betas, tbl["p"].tolist())):
        stars = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
        ax.text(b + max(errs[1]) * 0.05, i, f" {stars}", va="center", fontsize=10)

plt.suptitle("Surprisal β-Coefficient per Model\n(controlled for word length, frequency, position)",
             fontsize=13)
plt.tight_layout()
path = os.path.join(FIG_DIR, "fig_nb4_beta_forest.png")
plt.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", path)


In [ ]:
# ── 4.2 R² comparison across models ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (corpus, tbl) in zip(axes, tables.items()):
    bars = ax.bar(tbl["Model"], tbl["R²_adj"],
                  color=[colors[corpus]] * len(tbl), alpha=0.85, edgecolor="white")
    for bar, val in zip(bars, tbl["R²_adj"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
                f"{val:.4f}", ha="center", va="bottom", fontsize=9)
    ax.set_ylabel("Adjusted R²")
    ax.set_title(f"{corpus} — Adj. R² (surprisal + controls)")
    ax.tick_params(axis="x", rotation=20)

plt.suptitle("Model Fit: Adjusted R² (surprisal + word_len + log_freq + position)",
             fontsize=12)
plt.tight_layout()
path = os.path.join(FIG_DIR, "fig_nb4_r2.png")
plt.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", path)


---
## Part 5: Added-Variable (Partial Regression) Plots
Partial regression plots isolate the unique linear contribution of each surprisal model  
after removing the shared variance with word length, frequency, and position.  
If the surprisal effect is real and not driven by confounds, these plots should show a  
clear positive slope.


In [ ]:
# ── 5.1 Added-variable plot: bigram surprisal on Natural Stories ──────────────
from statsmodels.graphics.regressionplots import plot_partregress_grid

for model_name, col in MODEL_COLS[:2]:    # bigram + trigram shown as demo
    sub   = ns_full[["mean_log_RT", col] + CONTROLS_NS].dropna()
    formula = f"mean_log_RT ~ {col} + " + " + ".join(CONTROLS_NS)
    res   = smf.ols(formula, data=sub).fit()
    fig   = plt.figure(figsize=(14, 4))
    plot_partregress_grid(res, fig=fig)
    fig.suptitle(f"Added-Variable Plots — {model_name} (Natural Stories)", fontsize=12)
    plt.tight_layout()
    safe = model_name.lower().replace(" ", "_").replace("(","").replace(")","")
    path = os.path.join(FIG_DIR, f"fig_nb4_partregress_{safe}.png")
    plt.savefig(path, dpi=130, bbox_inches="tight")
    plt.show()
    print(f"Saved: {path}")


In [ ]:
# ── 6. Summary printout ───────────────────────────────────────────────────────
print("=" * 70)
print("NOTEBOOK 4 — SUMMARY")
print("=" * 70)
print()
print("Natural Stories — β_surprisal (per bit of surprisal):")
for _, row in ns_ols_table.iterrows():
    sig = "***" if row["p"] < 0.001 else ("**" if row["p"] < 0.01 else "*")
    print(f"  {row['Model']:14s}  β={row['β_surprisal']:+.5f}  "
          f"SE={row['SE']:.5f}  t={row['t']:+.2f}  p={row['p']:.4f} {sig}")
print()
print("Dundee — β_surprisal:")
for _, row in du_ols_table.iterrows():
    sig = "***" if row["p"] < 0.001 else ("**" if row["p"] < 0.01 else "*")
    print(f"  {row['Model']:14s}  β={row['β_surprisal']:+.5f}  "
          f"SE={row['SE']:.5f}  t={row['t']:+.2f}  p={row['p']:.4f} {sig}")
print()
print("Key interpretation:")
print("  A positive β means: each additional bit of surprisal increases log RT")
print("  by β units, after holding word length, frequency and position constant.")
print("  All models are expected to have β > 0 (surprisal theory prediction).")
